## 0. Beam extraction and plotting

First we import the relevant python packages:

In [6]:
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util.shape

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

## 1. Extracting beam geometry

The following code imports the model and extracts the beam geometry. To change the model, edit the `model_name` parameter to the relevant file name. Ensure the file is in the current directory. The beam geometry is saved in `beam_centers`, `beam_starts` and `beam_ends`.

In [ ]:

model_name = "26_02_D_STR.ifc" # 26-01-D-STR.ifc
# Load IFC
model = ifcopenshell.open(model_name)
settings = ifcopenshell.geom.settings()

# Extract beams
beams = model.by_type("IfcBeam")

# Initialize arrays
beam_centers = []
beam_starts = []
beam_ends = []

for beam in beams:

    try:
        shape = ifcopenshell.geom.create_shape(settings, beam)

        # Global vertex coordinates
        vertices = np.array(ifcopenshell.util.shape.get_shape_vertices(shape, shape.geometry))

        # Find main beam direction using PCA
        center = vertices.mean(axis=0)
        U, S, v = np.linalg.svd(vertices - center)

        direction = v[0]
        projections = (vertices - center) @ direction

        # Find endpoints
        start = center + projections.min() * direction
        end = center + projections.max() * direction

        # Store points
        beam_centers.append(center)
        beam_starts.append(start)
        beam_ends.append(end)

    except Exception as e:
        print(f"Could not process beam {beam.Name}: {e}")


# Convert to numpy arrays
beam_centers = np.array(beam_centers)
beam_starts = np.array(beam_starts)
beam_ends = np.array(beam_ends)

print(f"There are {len(beams)} beams in this model")

There are 630 beams in this model


## 2. Plotting all beams

The following code plots every beam in the model as line geometry. The centers of the beams are shown with a blue point.

In [ ]:
# Create interactive 3D figure
fig = go.Figure()

# Plot all beams as lines
for start, end in zip(beam_starts, beam_ends):

    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines", line=dict(color="red", width=5), showlegend=False))

# Plot beam centers
if len(beam_centers) > 0:
    fig.add_trace(go.Scatter3d(
        x=beam_centers[:, 0],
        y=beam_centers[:, 1],
        z=beam_centers[:, 2],
        mode="markers", marker=dict(size=3, color="blue"), name="Beam Centers"))

# Layout settings
fig.update_layout(
    title="Interactive 3D IFC Beam Model",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"), width=1200, height=900)

# Display interactive plot
fig.show()

---
## Vertices

In [ ]:
# Choose which beam you want to look at
beam = beams[300] #468, 300

shape = ifcopenshell.geom.create_shape(settings, beam)

# Global vertex coordinates
vertices = np.array(ifcopenshell.util.shape.get_shape_vertices(shape, shape.geometry))

# Find main beam direction using PCA
center = vertices.mean(axis=0)
U, S, v = np.linalg.svd(vertices - center)

direction = v[0]
projections = (vertices - center) @ direction

# Find endpoints
start = center + projections.min() * direction
end = center + projections.max() * direction

centroid = np.array(ifcopenshell.util.shape.get_shape_bbox_centroid(shape, shape.geometry))
print(centroid)

# Create interactive 3D figure
fig = go.Figure()

# Plot all vertices
fig.add_trace(go.Scatter3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    mode="markers",
    marker=dict(color="red", size=3),
    name="Vertices"
    ))

# Plot first principal direction
fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]],
        y=[start[1], end[1]],
        z=[start[2], end[2]],
        mode="lines",
        line=dict(
            color="blue",
            width=5
        ),
        showlegend=False
    ))

# Layout settings
fig.update_layout(
    title="Vertices of an IfcBeam",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
    width=1200,
    height=900
)

# Display interactive plot
fig.show()

[ 56.05018925 -19.37979422   0.415     ]
